<div class="alert alert-info" style="font-family:'arial';font-size:25px"> How to work with Variant Annotation Table (VAT) </div>

In this notebook, we will demonstrate how to extract variant info from the VAT using Hail, given a gene symbol.

Please be aware that this process takes a significant time and we recommend using a background notebook to run it. 

Runtime estimation: using the dataproac cluster setting (2/2 workers), it takes around 7-8 hrs to finish. Using 2/100 workers, n2-standard-8 machine type, it takes about 5-10mins.

More info about VAT is here, https://support.researchallofus.org/hc/en-us/articles/4615256690836-Variant-Annotation-Table.

In [ ]:
from datetime import datetime
start = datetime.now()
start

In [ ]:
%run /home/dataproc/workspace/aou-tutorial-notebooks/Setting_Env_Variables.ipynb

In [ ]:
import os
bucket=os.getenv("WORKSPACE_BUCKET")
bucket

In [ ]:
bucket = %env WORKSPACE_BUCKET
bucket

In [ ]:
google_project_id = %env GOOGLE_CLOUD_PROJECT
google_project_id

In [ ]:
import hail as hl
hl.init(
    gcs_requester_pays_configuration=google_project_id
)

In [ ]:
hl.default_reference(new_default_reference = "GRCh38")

# Where is the VAT?

In [ ]:
#VAT can be found here
!gcloud storage ls --billing-project=$GOOGLE_PROJECT gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/vat

In [ ]:
auxiliary_path = "gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux"
auxiliary_path

In [ ]:
vat_path = f'{auxiliary_path}/vat/*.gz'
vat_path

**Check the file size**

In [ ]:
#VAT can be found here
!gcloud storage ls -l --billing-project=$GOOGLE_PROJECT gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/vat

**Using Hail to import VAT**

In [ ]:
%%time
vat_table = hl.import_table(vat_path, force=True, quote='"', delimiter="\t", force_bgz=True)

# Filter the VAT by gene symbol

**Filter the VAT by gene symbol**

In [ ]:
gene='BRCA1'

In [ ]:
vat_new = vat_table.filter(vat_table["gene_symbol"]==gene)

**Only select the fields you need**

In [ ]:
vat_new=vat_new.select('vid', 'contig','position','genomic_location','gene_symbol', 'dbsnp_rsid','ref_allele',
'alt_allele','consequence','clinvar_classification','clinvar_phenotype','variant_type','gvs_all_ac','gvs_all_an','gvs_all_af')

In [ ]:
vat_new.describe()

In [ ]:
# don't do this, will take at least 2 hours, 
#vat_new.show()

**Save the result to the bucket**

In [ ]:
out_path = f'{bucket}/data/genomics/vat_gene_{gene}_v8_hail.tsv'
out_path

In [ ]:
%%time
# save to the bucket, will take time
vat_new.export(out_path) 

**Check saved file in the bucket**

In [ ]:
!gcloud storage ls -l {bucket}/data/genomics/vat_gene_{gene}_v8_hail.tsv

In [ ]:
import pandas as pd
df=pd.read_csv(out_path,sep='\t')
df.shape

In [ ]:
df.head()

In [ ]:
# total time
end = datetime.now()
end-start